# 12 - Best Track Hindcast: Damage Summary by Region

Summary statistics of median predicted houses damaged per region across all hindcast storms (2006-2024).
Based on per-storm damage tables from the besttrack hindcast.

In [1]:
import io, sys
from itertools import product as iproduct
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import ocha_stratus as stratus
from IPython.display import display
from src.constants import CHD_GREEN

plt.rcParams.update({"font.family": "sans-serif"})

In [2]:
CONTAINER    = "projects"
STAGE        = "dev"
PREFIX       = "pa-aa-phl-storms/processed/"
RECORD_YEARS = 19
TARGET_RP    = 3          # 1-in-3-year return period
TARGET_SEAS  = round(RECORD_YEARS / TARGET_RP)   # = 6

TARGET_REGIONS = ["Region II", "Region III", "Region V", "Region VIII"]

# Calibrated damage thresholds (houses) per region and tier.
# Source: binary 500-increment hindcast flags, combined 2006-2024 dataset.
# Constraint: RII > max(RV, RVIII) and RIII > max(RV, RVIII) at every tier.
# Tier 3 is the target action tier (each region individually achieves RP ~3 yr).
REGION_THRESHOLDS = {
    "Region II":   [ 2_000,  4_000,  6_000,  9_000, 21_000],
    "Region III":  [ 2_000,  4_000,  6_000,  9_000, 21_000],
    "Region V":    [ 1_500,  3_500,  5_000,  6_000, 20_500],
    "Region VIII": [   500,  1_500,  2_500,  4_000,  7_500],
}
N_TIERS = 5

REGION_COLORS = {
    "Region II":   "#1565C0",
    "Region III":  "#E65100",
    "Region V":    "#6A1B9A",
    "Region VIII": "#B71C1C",
}

def _fmt(v):
    return "∞" if v >= 99 else f"{v:.1f}"

TBL_STYLES = [
    {"selector": "caption", "props": "text-align:left;padding:4px 0 10px 0;"},
    {"selector": "th",      "props": "background:#f5f5f5;padding:7px 14px;"
                                     "font-weight:600;text-align:center;"},
    {"selector": "td",      "props": "padding:6px 14px;text-align:center;"},
]

In [3]:
import io

# Three runs combined; corrected is the primary source.
# Storms not in corrected are supplemented from the updated run, then the old run.
COR_BLOB = PREFIX + "all_storms_hindcast_results/besttrack_500increments_corrected/all_besttrack_500increment_corrected_raw.csv"
NEW_BLOB = PREFIX + "all_storms_hindcast_results/besttrack_500increments/all_besttrack_500increment_raw.csv"
OLD_BLOB = PREFIX + "besttrack_hindcast_results/fine_500_increments/besttrack_500increment_raw.csv"

def _load_flag_csv(blob):
    raw = stratus.load_blob_data(blob, stage=STAGE, container_name=CONTAINER)
    df  = pd.read_csv(io.BytesIO(raw))
    if "region" in df.columns and "Region" not in df.columns:
        df = df.rename(columns={"region": "Region"})
    df["storm"] = df["storm"].str.upper()
    return df

cor_df = _load_flag_csv(COR_BLOB)
new_df = _load_flag_csv(NEW_BLOB)
old_df = _load_flag_csv(OLD_BLOB)

THR_COLS = [c for c in cor_df.columns if str(c).isdigit()]
keep_cols = ["storm", "year", "Region"] + THR_COLS

def _missing(df, known_keys):
    return df[
        df.apply(lambda r: (r["storm"], int(r["year"])) not in known_keys, axis=1)
    ]

cor_keys = set(zip(cor_df["storm"], cor_df["year"].astype(int)))
new_only = _missing(new_df, cor_keys)

combined_keys = cor_keys | set(zip(new_only["storm"], new_only["year"].astype(int)))
old_only = _missing(old_df, combined_keys)

raw_df = pd.concat(
    [cor_df[[c for c in keep_cols if c in cor_df.columns]],
     new_only[[c for c in keep_cols if c in new_df.columns]],
     old_only[[c for c in keep_cols if c in old_df.columns]]],
    ignore_index=True,
).sort_values(["year", "storm", "Region"]).reset_index(drop=True)

n_cor   = cor_df[["storm", "year"]].drop_duplicates().shape[0]
n_new   = new_only[["storm", "year"]].drop_duplicates().shape[0]
n_old   = old_only[["storm", "year"]].drop_duplicates().shape[0]
n_total = raw_df[["storm", "year"]].drop_duplicates().shape[0]

print(f"{n_total} storms total  "
      f"({n_cor} corrected + {n_new} updated + {n_old} old)")
print(f"Years: {int(raw_df['year'].min())}-{int(raw_df['year'].max())}  "
      f"| target RP {TARGET_RP} yr = {TARGET_SEAS} seasons in {RECORD_YEARS} yr")


def triggered_years(region, threshold):
    """Return set of years where the binary trigger flag = 1."""
    col = str(int(threshold))
    sub = raw_df[(raw_df["Region"] == region) & (raw_df[col] == 1.0)]
    return set(sub["year"].astype(int).unique())


def triggered_storms(region, threshold):
    """Return rows where the binary trigger flag = 1, sorted by year."""
    col = str(int(threshold))
    return raw_df[(raw_df["Region"] == region) & (raw_df[col] == 1.0)].sort_values("year")

89 storms total  (74 corrected + 1 updated + 14 old)
Years: 2006-2024  | target RP 3 yr = 6 seasons in 19 yr


In [5]:
# ── Summary statistics of median predicted damage per region ───────────────────
# Loads per-storm damage tables from blob (separate from binary flag CSV above)
DAMAGE_PREFIX = PREFIX + "besttrack_hindcast_results/per_storm_output/"

damage_blobs = [
    b for b in stratus.list_container_blobs(
        name_starts_with=DAMAGE_PREFIX, stage=STAGE, container_name=CONTAINER
    )
    if b.endswith("_damage_table.csv")
]

dmg_records = []
for blob in damage_blobs:
    key        = blob.replace(DAMAGE_PREFIX, "").replace("_damage_table.csv", "")
    storm_name = key.rsplit("_", 1)[0].upper()
    year       = int(key.rsplit("_", 1)[1][:4])
    raw        = stratus.load_blob_data(blob, stage=STAGE, container_name=CONTAINER)
    df         = pd.read_csv(io.BytesIO(raw))
    by_region  = df.groupby("adm1_en")["median"].sum()
    for region in TARGET_REGIONS:
        dmg_records.append({
            "storm":  storm_name,
            "year":   year,
            "region": region,
            "damage": int(round(by_region.get(region, 0))),
        })

dmg_df  = pd.DataFrame(dmg_records)
nonzero = dmg_df[dmg_df["damage"] > 0].copy()

pct_cols = [10, 25, 50, 75, 90, 95]
stats_rows = []
for region in TARGET_REGIONS:
    sub = nonzero[nonzero["region"] == region]["damage"]
    row = {"Region": region, "N storms": len(sub),
           "Min": int(sub.min()), "Max": int(sub.max())}
    for p in pct_cols:
        row[f"P{p}"] = int(np.percentile(sub, p))
    stats_rows.append(row)

stats_df = pd.DataFrame(stats_rows).set_index("Region")

display(
    stats_df.style
    .format(lambda v: f"{int(v):,}")
    .background_gradient(cmap="YlOrRd", axis=None,
                         subset=[f"P{p}" for p in pct_cols] + ["Max"])
    .set_caption(
        "<b style='font-size:14px;color:#2e7d32'>"
        "Median predicted houses damaged - summary by region</b>"
        "<br><span style='color:#555;font-size:11px'>"
        "Storms with zero predicted damage excluded; values from per-storm best track damage tables"
        "</span>"
    )
    .set_table_styles(TBL_STYLES)
)

,N storms,Min,Max,P10,P25,P50,P75,P90,P95
Region,,,,,,,,,
Region II,27,1,"14,168",6,112,333,"2,786","6,822","9,850"
Region III,30,3,"20,621",4,35,937,"4,010","13,843","17,228"
Region V,25,31,"113,509",54,315,"3,601","25,190","69,767","78,481"
Region VIII,19,14,"203,620",32,94,"2,453","16,246","24,807","44,601"
